In [1]:
import pandas as pd
import json

In [2]:
# 1. Read the JSON file using standard Python
with open('../data/raw/lahore_aqi_raw.json', 'r') as f:
    raw_data = json.load(f)

# 2. Extract ONLY the 'hourly' key where the actual data arrays live
df = pd.DataFrame(raw_data['hourly'])

# 3. Take a look at your clean DataFrame
print(df.head())

               time  pm2_5   pm10  us_aqi
0  2026-01-01T00:00  106.9  107.8     230
1  2026-01-01T01:00   99.7  101.0     228
2  2026-01-01T02:00   94.1   95.7     226
3  2026-01-01T03:00   90.3   92.0     223
4  2026-01-01T04:00   81.3   83.1     220


In [3]:
# Converting 'time' string to actual time object
df['time'] = pd.to_datetime(df['time'])

df['date'] = df['time'].dt.date
df['time'] = df['time'].dt.time

In [4]:
df.set_index(['date', 'time'], inplace=True)

In [5]:
df.tail(72)   # 3 Days

pm2_5  pm10  us_aqi
date       time                         
2026-10-09 00:00:00   63.8  73.6     166
           01:00:00   62.6  70.3     166
           02:00:00   61.9  68.5     166
           03:00:00   61.9  69.6     165
           04:00:00   61.0  67.8     165
...                    ...   ...     ...
2026-10-11 19:00:00   19.5  21.8     127
           20:00:00   23.6  26.1     124
           21:00:00   25.9  28.6     120
           22:00:00   27.6  30.5     116
           23:00:00   28.7  31.8     112

[72 rows x 3 columns]

## **Defining Rules**

**Defining Some Rules of this Weather AQI Application (Based on Three Values : US_AQI, PM2_5, PM10)**


| Category | US AQI Score | PM2.5 (μg/m³) | PM10 (μg/m³) | What it means in real life |
| :--- | :--- | :--- | :--- | :--- |
| **Good (Green)** | 0 – 50 | 0 – 12 | 0 – 54 | Clean air. Great day to be outside. |
| **Moderate (Yellow)** | 51 – 100 | 12.1 – 35.4 | 54.1 – 154 | Acceptable air quality. |
| **Unhealthy for Sensitive Groups (Orange)** | 101 – 150 | 35.5 – 55.4 | 154.1 – 254 | People with asthma or elderly may feel irritation. |
| **Unhealthy (Red)** | 151 – 200 | 55.5 – 150.4 | 254.1 – 354 | Everyone begins to feel throat irritation or coughs. |
| **Very Unhealthy (Purple)** | 201 – 300 | 150.5 – 250.4 | 354.1 – 424 | Health alert. Avoid outdoor activities. |
| **Hazardous (Maroon)** | 301 – 500+ | 250.5+ | 424.1+ | Emergency conditions. Serious risk for whole population. |


### **What App Gonna do**

App have to print this :

<pre style="background-color: #1e1e1e; color: #d4d4d4; padding: 15px; border-radius: 5px; font-family: monospace; line-height: 1.5;">
==================================================================
                    <b>LAHORE AIR QUALITY REPORT</b>                     
==================================================================

<b>[CURRENT STATUS]</b> (2026-10-04 23:00)
 - US AQI : <span style="color: #ff6b6b;"><b>312 [HAZARDOUS]</b></span>
 - PM2.5  : 262.4 µg/m³
 - PM10   : 310.1 µg/m³

<b>[HISTORICAL TRENDS]</b>
 - Past 3 Days Avg  : 285 AQI
 - Past 7 Days Avg  : 240 AQI
 - Past 30 Days Avg : 180 AQI
 -> <span style="color: #ffaa44;">Trend: Air quality has <b>WORSENED by 18.7%</b> over the last week.</span>

<b>[PATTERN INSIGHTS]</b>
 - Safest Window : <span style="color: #51cf66;">2:00 PM - 4:00 PM</span> (Avg AQI: 160)
 - Peak Danger   : <span style="color: #ff6b6b;">11:00 PM - 4:00 AM</span>  (Avg AQI: 340)

==================================================================
</pre>


## **Calculation And Processing**

### **Defining Terms**

In [7]:
def get_aqi_meaning(aqi_value):
    if aqi_value <= 50:
        return "Good"
    elif aqi_value <= 100:
        return "Moderate"
    elif aqi_value <= 150:
        return "Unhealthy for Sensitive Groups"
    elif aqi_value <= 200:
        return "Unhealthy"
    elif aqi_value <= 300:
        return "Very Unhealthy"
    else:
        return "Hazardous"

### **Current Status**

In [17]:
# Get todays date in <Year-Month-Date> format
from datetime import date, datetime
today = date.today()
print(today)

# Finding current hour
hour = datetime.now().hour
print(hour)

2026-10-11
22


In [20]:
# Getting current's status
aqi, pm2_5, pm10 = df.loc[today].iloc[hour].round(2)[['us_aqi', 'pm2_5', 'pm10']]

print(aqi, pm2_5, pm10)


128.0 57.3 63.6


### **Historical Status**

In [28]:
past_3 = df.tail(24*3)['us_aqi'].mean().round(2)
past_7 = df.tail(24*7)['us_aqi'].mean().round(2)
past_30 = df.tail(24*30)['us_aqi'].mean().round(2)

last_week = df.iloc[-24*14:-24*7]['us_aqi'].mean().round(2)

print(past_7, last_week)

150.74 157.89


In [34]:
if past_7 > last_week :
  print(round(((past_7 - last_week) / past_7) * 100, 2))
else :
  print(round(((last_week - past_7) / last_week) * 100, 2))

4.53


In [ ]:
past_aqi = [df.tail(24*i)['us_aqi'].mean().round(2) for i in [3, 7, 30]]

[np.float64(149.99), np.float64(150.74), np.float64(156.72)]


### **Pattern**

In [ ]:
temp = df.tail(24*7)['us_aqi'].groupby('time').mean().sort_values()

In [44]:
lowest = temp.head(4)
highest = temp.tail(4)

lowest

time
22:00:00    145.571429
23:00:00    145.714286
15:00:00    147.142857
14:00:00    147.428571
Name: us_aqi, dtype: float64

In [45]:
highest

time
20:00:00    155.285714
17:00:00    156.714286
19:00:00    159.857143
18:00:00    161.142857
Name: us_aqi, dtype: float64

In [46]:
print(lowest.mean(), highest.mean())

146.46428571428572 158.25


### Safest Window

In [6]:
temp = df.tail(72)['us_aqi'].groupby('time').mean().sort_values()
lowest = temp.head(4)
highest = temp.tail(4)

In [14]:
# print(i.strftime('%I:%M %p') for i in lowest.index)
for i in lowest.index:
  i = i.strftime('%I:%M %p')
  print(i)

11:00 PM
10:00 PM
09:00 PM
08:00 PM


In [8]:
highest

time
03:00:00    155.333333
04:00:00    156.000000
06:00:00    156.000000
05:00:00    156.333333
Name: us_aqi, dtype: float64